# NuNER Dataset Preprocessing

Load and preprocess the [NuNER](https://huggingface.co/datasets/numind/NuNER) dataset for information extraction evaluation.

In [ ]:
import ast
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset, DatasetDict

## Load Dataset

In [ ]:
ds = load_dataset("numind/NuNER", split="full")
print(f"Loaded {len(ds):,} rows")
print(f"Columns: {ds.column_names}")

## Basic EDA

In [ ]:
df = ds.to_pandas()
print(f"Shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum():,}")
df.head()

## Parse Output Column

Convert `"['entity <> type', ...]"` strings into structured lists of `{"entity": ..., "type": ...}` dicts.

In [ ]:
def parse_entities(output_str: str) -> list[dict]:
    """Parse an output string into a list of {entity, type} dicts."""
    try:
        items = ast.literal_eval(output_str)
    except (ValueError, SyntaxError):
        return []
    entities = []
    for item in items:
        parts = item.split(" <> ", maxsplit=1)
        if len(parts) == 2:
            entities.append({"entity": parts[0].strip(), "type": parts[1].strip()})
    return entities


# Test on a few rows
for i in range(3):
    print(f"Input:  {df.iloc[i]['input'][:100]}")
    print(f"Parsed: {parse_entities(df.iloc[i]['output'])}")
    print()

## Entity Type Analysis

In [ ]:
type_counter = Counter()
parse_failures = 0

for output_str in df["output"]:
    parsed = parse_entities(output_str)
    if not parsed and output_str != "[]":
        parse_failures += 1
    for ent in parsed:
        type_counter[ent["type"]] += 1

print(f"Unique entity types: {len(type_counter)}")
print(f"Parse failures: {parse_failures:,}")
print(f"Total entities: {sum(type_counter.values()):,}")
print(f"\nTop 20 types:")
for etype, count in type_counter.most_common(20):
    print(f"  {etype:30s} {count:>10,}")

In [ ]:
top20 = type_counter.most_common(20)
labels, counts = zip(*top20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(labels)), counts)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.set_title("Top 20 Entity Types in NuNER")
plt.tight_layout()
plt.show()

## Train / Val / Test Split (80/10/10)

In [ ]:
SEED = 42

# First split: 80% train, 20% temp
split1 = ds.train_test_split(test_size=0.2, seed=SEED)
# Second split: 50/50 on the 20% temp -> 10% val, 10% test
split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)

splits = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})

for name, split in splits.items():
    print(f"{name:12s}: {len(split):>10,} rows")

## Save to Disk

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

for name, split in splits.items():
    out_path = data_dir / f"{name}.parquet"
    split.to_parquet(str(out_path))
    print(f"Saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

print("\nDone!")

In [ ]:
# Save generic and domain training subsets
keep_cols = ["input", "output"]

generic_df = train_df[~train_df["is_domain"]][keep_cols].reset_index(drop=True)
domain_df  = train_df[ train_df["is_domain"]][keep_cols].reset_index(drop=True)

generic_path = data_dir / "generic_train.parquet"
domain_path  = data_dir / "domain_train.parquet"

generic_df.to_parquet(str(generic_path), index=False)
domain_df.to_parquet(str(domain_path),  index=False)

print(f"Saved {generic_path}  ({len(generic_df):,} rows, {generic_path.stat().st_size/1e6:.1f} MB)")
print(f"Saved {domain_path}   ({len(domain_df):,} rows, {domain_path.stat().st_size/1e6:.1f} MB)")
print("\nAll data splits ready for fine-tuning notebooks.")

In [ ]:
# Top entity types in domain vs. generic splits
domain_types  = Counter()
generic_types = Counter()

for _, row in train_df.iterrows():
    counter = domain_types if row["is_domain"] else generic_types
    for ent in row["parsed_entities"]:
        counter[ent["type"]] += 1

print("Top 15 entity types — Domain (tech) samples:")
for t, c in domain_types.most_common(15):
    print(f"  {t:35s} {c:>8,}")

print("\nTop 15 entity types — Generic samples:")
for t, c in generic_types.most_common(15):
    print(f"  {t:35s} {c:>8,}")

In [ ]:
# Keywords that signal a technology/software domain entity type (case-insensitive substring match)
TECH_TYPE_KEYWORDS = [
    "software", "algorithm", "programming", "database", "framework",
    "library", "api", "neural", "machine learning", "deep learning",
    "artificial intelligence", "operating system", "hardware", "processor",
    "network", "protocol", "data structure", "platform", "tool",
    "application", "service", "cloud", "code", "function", "class",
    "module", "package", "repository", "dataset", "benchmark",
    "technology", "computer", "computing", "digital", "cyber",
    "internet", "web", "mobile", "device", "sensor", "chip",
    "version", "release", "language", "compiler", "runtime",
    "virtual", "container", "kubernetes", "docker", "linux",
    "interface", "server", "client", "endpoint", "pipeline",
]


def is_tech_domain(entities: list[dict]) -> bool:
    """Return True if any entity type in the sample suggests a tech domain."""
    for ent in entities:
        etype = ent["type"].lower()
        if any(kw in etype for kw in TECH_TYPE_KEYWORDS):
            return True
    return False


# Classify each training sample
train_df = splits["train"].to_pandas()
train_df["parsed_entities"] = train_df["output"].map(parse_entities)
train_df["is_domain"] = train_df["parsed_entities"].map(is_tech_domain)

n_domain  = train_df["is_domain"].sum()
n_generic = (~train_df["is_domain"]).sum()
print(f"Training set size : {len(train_df):,}")
print(f"  Domain (tech)   : {n_domain:,}  ({n_domain/len(train_df)*100:.1f}%)")
print(f"  Generic         : {n_generic:,}  ({n_generic/len(train_df)*100:.1f}%)")

## Generic vs. Domain (Tech) Sample Classification

Each training sample is classified as **generic** or **domain (tech)** based on whether any of its
entity types match a set of technology-related keywords.  
The two subsets are saved as `generic_train.parquet` and `domain_train.parquet` for use in the
fine-tuning notebooks.